
Подразумевался как файл, для преодобработки коротких аудиозаписей (EDA буквально)


In [ ]:
import os
from os import listdir
from os.path import join, basename

In [ ]:
'''
так вот, первое что пришло в голову, так это работать с train_audio, чтобы пройтись по каждому классу, в каждом классе пройтись по каждой записи, каждую запись нарезать по 5 секундные куски

обучаем resnet и последний слой можно использовать как среднее представление класса (некий латентый вектор)

потом (ситуация теста, где мне нужно классифицировать), просто взять, нарезать аудио так же по 5 секунд, закинуть в уже обученный resnet, вытащить латентый вектор, сравнить с косинусным расстоянием со средними представлениями для всех классов, считаем сходство со всеми классами и усредняем вероятности по всем кускам файла.

классов много. Можно юзнуть arcface
'''

'''
1 Нарезка 5 сек.
2 ResNet + ArcFace Head.
3 Инференс: Эмбеддинг на Веса слоя Потом Сигмоида Потом Усреднение по кускам файла
'''

In [ ]:
path = r'data/train_audio'

folders = listdir(path)     # 206 классов в коротких
pathes = {}                 # pathes[class] = [path1, path2, ...]

for ind,folder in enumerate(folders):

    folder_path = join(path, folder)
    pathes[folder] = [join(folder_path, file) for file in listdir(folder_path)]

In [ ]:
pathes['fabwre1'][:3]

In [ ]:
import librosa
import numpy as np
import torch

In [ ]:
def scaler(mel):

    log_mel = librosa.power_to_db(mel, ref=np.amax)
    min_, max_ = log_mel.min(), log_mel.max()
    
    if max_ - min_ == 0: return np.zeros_like(log_mel)

    return (log_mel - min_) / (max_ - min_)

def get_sps(y, sr: int):

    mel_low = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=2048, hop_length=512,
        n_mels=128, fmin=0, fmax=10_000
    )
    mel_high = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=2048, hop_length=512,
        n_mels=128, fmin=8_000, fmax=sr//2
    )
    stacked = np.stack(list(map(scaler, (mel_low, mel_high))), axis=0)
    return torch.tensor(stacked, dtype=torch.float32)

sr = 32_000
duration = 5
stride = 2.5

slide_duration = int(duration * sr)
slide_stride = int(stride * sr)

new_data_path = 'new_data/train_audio_sps'
os.makedirs(new_data_path, exist_ok=True)

for folder in list(pathes.keys()):

    count, all_slides = 0, 0
    os.makedirs(join(new_data_path, folder), exist_ok=True)

    for path in pathes[folder]:
        name = basename(path).split('.')[0]
        y, _ = librosa.load(path, sr=sr, mono=True)
        y_len = len(y)

        edge = 0
        for i in range(0, y_len, slide_stride):
            segment = y[i : i + slide_duration]
            
            if len(segment) < slide_duration:
                pad_width = slide_duration - len(segment)
                slide = np.pad(segment, (0, pad_width), mode='constant')
            else:
                slide = segment

            sps = get_sps(slide, sr)
            save_path = join(new_data_path, folder, f'{name}_{edge}.pt')
            torch.save(sps, save_path)

            edge += stride
            all_slides += 1
        count += 1

    print(f' Class [{folder:^7}] - complete    num_audio: {count},  all_slides: {all_slides}')